# Proteomics-Toolkit Tutorial

This tutorial walks through a complete analysis using the bundled example
dataset: a small, simulated Skyline-PRISM experiment with two groups
(`Control` vs `Treatment`), six samples per group, and 80 proteins with 60
peptides across the 20 most abundant proteins.

**Pipeline:**
1. Load the example dataset (protein + peptide + metadata)
2. Run QC plots (missing values, identifications, intensity distributions, CV)
3. Filter and normalise
4. Run an unpaired differential analysis
5. Visualise results (volcano, PCA)
6. Inspect peptide coverage for a specific protein
7. Export results

## 1. Load the example dataset

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import proteomics_toolkit as ptk

protein_data, peptide_data, metadata, sample_cols = ptk.datasets.load_example_data()
print(f"protein_data: {protein_data.shape}")
print(f"peptide_data: {peptide_data.shape}")
print(f"metadata:     {metadata.shape}")
print(f"sample_cols:  {len(sample_cols)} samples")
metadata

The PRISM batch-suffix convention (`__@__Batch1`) is baked into the sample
column names. Build a dict mapping each full sample column to its metadata
row so the stats functions can look up group membership.

In [ ]:
col_map = ptk.strip_batch_suffix(sample_cols)
short_to_col = {v: k for k, v in col_map.items()}

meta_dict = {}
for _, row in metadata.iterrows():
    full_col = short_to_col.get(row['Replicate'])
    if full_col:
        meta_dict[full_col] = row.to_dict()

list(meta_dict.items())[:2]

## 2. Quality-control plots

Inspect data quality before any filtering or normalization.

In [ ]:
ptk.plot_identifications_per_sample(protein_data, sample_cols, sample_metadata=meta_dict)
plt.show()

In [ ]:
ptk.plot_missing_value_heatmap(protein_data, sample_cols)
plt.show()

In [ ]:
ptk.plot_intensity_distributions(protein_data, sample_cols)
plt.show()

In [ ]:
ptk.plot_cv_distribution(protein_data, sample_cols, sample_metadata=meta_dict)
plt.show()

## 3. Build the standardized annotation + sample table

The toolkit's statistical and visualization functions expect a DataFrame
with the 5 standard annotation columns first, then sample columns.

In [ ]:
annot = protein_data[[
    'leading_protein', 'leading_description', 'leading_gene_name',
    'leading_uniprot_id', 'leading_name',
]].copy()
annot.columns = ['Protein', 'Description', 'Protein Gene', 'UniProt_Accession', 'UniProt_Entry_Name']

data = pd.concat(
    [annot.reset_index(drop=True), protein_data[sample_cols].reset_index(drop=True)],
    axis=1,
)
data.head()

## 4. Statistical analysis (unpaired Welch's t-test)

In [ ]:
config = ptk.StatisticalConfig()
config.analysis_type = 'unpaired'
config.statistical_test_method = 'welch_t'
config.group_column = 'Group'
config.group_labels = ['Control', 'Treatment']
config.correction_method = 'fdr_bh'
config.p_value_threshold = 0.05
config.fold_change_threshold = 1.0
config.log_transform_before_stats = True
config.validate()

results = ptk.run_comprehensive_statistical_analysis(
    data, meta_dict, config, protein_annotations=annot
)
results.head()

## 5. Volcano plot and PCA

In [ ]:
ptk.plot_volcano(results, fc_threshold=1.0, gene_column='Protein Gene', label_top_n=10)
plt.show()

In [ ]:
ptk.plot_pca(data, sample_cols, meta_dict)
plt.show()

## 6. Peptide coverage for a significant protein

Pick the most significant protein and inspect its peptide-level coverage
(this dataset includes peptides for the 20 most abundant proteins).

In [ ]:
top_protein = results.sort_values('P.Value').iloc[0]['Protein']
print(f"Top hit: {top_protein}")

# If the top protein has peptide-level data, plot its coverage
if (peptide_data['leading_protein'] == top_protein).any():
    ptk.plot_peptide_coverage_map(
        peptide_data,
        protein_id=top_protein,
        sample_columns=sample_cols,
        start_column='start_position',
    )
    plt.show()
else:
    print("Top hit has no peptide-level data in this bundled example.")

## 7. Export results

In [ ]:
exported = ptk.export_analysis_results(
    normalized_data=data,
    sample_metadata=meta_dict,
    differential_results=results,
    filtered_data=data,
    output_prefix='example_analysis',
)
exported

That's it: a full protein-level workflow on 80 proteins in a handful of cells.
Swap `protein_data` for `peptide_data` (and adjust annotation columns) to run
the same pipeline at the peptide level.